
# Landing to Bronze — V-Credit

## Objetivo

Ingerir os oito arquivos CSV de origem e armazená-los em tabelas Delta na camada Bronze do catálogo `vcredit`.

## Responsabilidades

- Criar e verificar a estrutura inicial do projeto.
- Conferir a presença dos arquivos de origem.
- Ler os CSVs com a codificação definida para cada arquivo.
- Preservar os valores de origem, mantendo as colunas como texto.
- Adicionar `ingestion_datetime` e `arquivo_origem` para rastreabilidade.
- Validar as quantidades de registros após a gravação.


In [0]:
# 1. Criar o catálogo do projeto
spark.sql("""
    CREATE CATALOG IF NOT EXISTS vcredit
""")

# 2. Criar os schemas das três camadas
for schema in ["bronze", "silver", "gold"]:
    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS vcredit.{schema}
    """)

# 3. Criar o volume de entrada na camada Bronze
spark.sql("""
    CREATE VOLUME IF NOT EXISTS vcredit.bronze.inputs
""")

# 4. Definir o caminho dos arquivos de origem
input_path = "/Volumes/vcredit/bronze/inputs/"

In [0]:
# 5. Listar os arquivos disponíveis
arquivos_disponiveis = dbutils.fs.ls(input_path)
display(arquivos_disponiveis)

# 6. Conferir se os oito arquivos esperados estão presentes
arquivos_esperados = [
    "base_atendentes.csv",
    "base_motivos.csv",
    "canais.csv",
    "chamados.csv",
    "Chamados_Hora.CSV",
    "clientes.csv",
    "custos.csv",
    "pesquisa_satisfacao.csv"
]

nomes_disponiveis = {
    arquivo.name for arquivo in arquivos_disponiveis
}

arquivos_faltantes = [
    nome for nome in arquivos_esperados
    if nome not in nomes_disponiveis
]

if arquivos_faltantes:
    raise FileNotFoundError(
        "Arquivos ausentes no volume de entrada: "
        + ", ".join(arquivos_faltantes)
    )

print("Todos os oito arquivos de origem foram encontrados.")


## Função de ingestão

A função `carregar_bronze` centraliza a leitura, as validações e a gravação dos arquivos na camada Bronze.

As colunas de origem são mantidas como texto. A função acrescenta metadados de rastreabilidade, grava os dados em formato Delta e compara as quantidades de registros lidos e gravados.

In [0]:
from pyspark.sql import functions as F


def carregar_bronze(nome_arquivo, nome_tabela, encoding):

    # 1. Montar o caminho completo do arquivo
    caminho = input_path + nome_arquivo

    print(f"Iniciando leitura: {nome_arquivo}")
    print(f"Codificação utilizada: {encoding}")

    # 2. Ler o arquivo CSV
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("sep", ",")
        .option("encoding", encoding)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(caminho)
    )

    # 3. Padronizar os nomes das colunas
    df = df.toDF(
        *[coluna.strip().lower() for coluna in df.columns]
    )

    # 4. Validar se o arquivo possui colunas
    if len(df.columns) == 0:
        raise ValueError(
            f"O arquivo {nome_arquivo} não possui colunas."
        )

    # 5. Validar nomes duplicados após a padronização
    if len(df.columns) != len(set(df.columns)):
        raise ValueError(
            f"O arquivo {nome_arquivo} possui "
            "nomes de colunas duplicados."
        )

    # 6. Conferir a estrutura e a quantidade de registros
    df.printSchema()

    quantidade_origem = df.count()

    print(
        f"Registros lidos de {nome_arquivo}: "
        f"{quantidade_origem}"
    )

    # 7. Adicionar metadados de rastreabilidade
    df = (
        df
        .withColumn(
            "ingestion_datetime",
            F.current_timestamp()
        )
        .withColumn(
            "arquivo_origem",
            F.lit(nome_arquivo)
        )
    )

    # 8. Gravar na tabela Bronze
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(nome_tabela)
    )

    # 9. Ler a tabela gravada para validar o resultado
    df_bronze = spark.table(nome_tabela)

    quantidade_bronze = df_bronze.count()

    print(f"Tabela carregada: {nome_tabela}")
    print(f"Registros de origem: {quantidade_origem}")
    print(f"Registros na Bronze: {quantidade_bronze}")

    # 10. Verificar se as quantidades correspondem
    if quantidade_origem != quantidade_bronze:
        raise ValueError(
            f"A quantidade de registros não corresponde "
            f"na tabela {nome_tabela}."
        )

    print("Carga concluída com sucesso.")

    return df_bronze



## Mapeamento dos arquivos de origem

Nesta etapa, cada CSV é associado à sua tabela Delta na camada Bronze e à codificação definida para a leitura.

O processamento é executado por um laço, reutilizando a função de ingestão para as oito bases.

In [0]:

# Arquivo de origem -> tabela Bronze -> codificação
arquivos_bronze = [
    (
        "chamados.csv",
        "vcredit.bronze.tb_chamados",
        "UTF-8"
    ),
    (
        "Chamados_Hora.CSV",
        "vcredit.bronze.tb_chamados_hora",
        "ISO-8859-1"
    ),
    (
        "clientes.csv",
        "vcredit.bronze.tb_clientes",
        "ISO-8859-1"
    ),
    (
        "custos.csv",
        "vcredit.bronze.tb_custos",
        "UTF-8"
    ),
    (
        "pesquisa_satisfacao.csv",
        "vcredit.bronze.tb_pesquisa_satisfacao",
        "UTF-8"
    ),
    (
        "canais.csv",
        "vcredit.bronze.tb_canais",
        "UTF-8"
    ),
    (
        "base_motivos.csv",
        "vcredit.bronze.tb_motivos",
        "UTF-8"
    ),
    (
        "base_atendentes.csv",
        "vcredit.bronze.tb_atendentes",
        "UTF-8"
    )
]

# Executar a carga de cada arquivo
for nome_arquivo, nome_tabela, encoding in arquivos_bronze:
    carregar_bronze(
        nome_arquivo,
        nome_tabela,
        encoding
    )



## Validação das tabelas Bronze

Após a carga, este bloco consulta as tabelas gravadas, exibe a quantidade de registros e apresenta uma amostra dos dados para inspeção.


In [0]:
# Verificar as tabelas carregadas na camada Bronze
for _, nome_tabela, _ in arquivos_bronze:
    df_bronze = spark.table(nome_tabela)

    print(f"\nTabela: {nome_tabela}")
    print(
        f"Quantidade de registros: {df_bronze.count()}"
    )

    display(df_bronze.limit(5))


## Verificação complementar da codificação

Esta etapa procura padrões de caracteres que podem indicar problemas de leitura ou codificação nas tabelas Bronze.

Os resultados são indícios para investigação e devem ser comparados com os arquivos de origem. Esta verificação não modifica nem corrige os dados.

In [0]:

# Verificar possíveis problemas de codificação nas tabelas Bronze

caracteres_suspeitos = ["�", "Ã§", "Ã£", "Ã¡", "Ã©", "Ã³", "Ãº", "Ã­", "Â"]

for _, nome_tabela, _ in arquivos_bronze:
    df = spark.table(nome_tabela)

    # Selecionar as colunas de texto
    colunas_texto = [
        campo.name
        for campo in df.schema.fields
        if campo.dataType.simpleString() == "string"
    ]

    # Procurar caracteres suspeitos em qualquer coluna de texto
    condicao = None

    for coluna in colunas_texto:
        for caractere in caracteres_suspeitos:
            teste = F.col(coluna).contains(caractere)
            condicao = teste if condicao is None else condicao | teste

    if condicao is not None:
        suspeitos = df.filter(condicao)
        quantidade = suspeitos.count()

        print(f"\nTabela: {nome_tabela}")
        print(f"Linhas com possíveis problemas: {quantidade}")

        if quantidade > 0:
            display(suspeitos.limit(5))
    else:
        print(f"\nTabela: {nome_tabela} - sem colunas de texto")
